In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
import warnings
import platform
import matplotlib.pyplot as plt

# 출력 설정
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# 한글 폰트 설정
import platform
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':  # macOS
    plt.rcParams['font.family'] = 'AppleGothic'
else:  # Linux
    plt.rcParams['font.family'] = 'NanumGothic'

plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (12, 6)

# 참고: seed 고정으로 팀원 간 동일한 결과 재현 가능
np.random.seed(42)

print("="*60)
print("라이브러리 로드 완료!")
print("한글 폰트 설정 완료!")
print("="*60)

라이브러리 로드 완료!
한글 폰트 설정 완료!


In [2]:
# 1. 원본 다시 읽기
df_portfolio = pd.read_csv("data/portfolio.csv", index_col=0)
df_profile = pd.read_csv("clean_data/profile_cleaned.csv", index_col=0)
df_transcript = pd.read_csv("clean_data/transcript_cleaned.csv", index_col=0)

print("=== 원본 컬럼 확인 ===")
print("profile:", df_profile.columns.tolist())
print("portfolio:", df_portfolio.columns.tolist())
print("transcript:", df_transcript.columns.tolist())
print()

# 2. 컬럼명 정리
# profile: id -> customer_id
if "id" in df_profile.columns:
    df_profile = df_profile.rename(columns={"id": "customer_id"})

# portfolio: id -> offer_id
if "id" in df_portfolio.columns:
    df_portfolio = df_portfolio.rename(columns={"id": "offer_id"})

# transcript: person -> customer_id (단, 이미 customer_id가 없을 때만)
if "customer_id" not in df_transcript.columns and "person" in df_transcript.columns:
    df_transcript = df_transcript.rename(columns={"person": "customer_id"})

print("=== 정리 후 컬럼 확인 ===")
print("profile:", df_profile.columns.tolist())
print("portfolio:", df_portfolio.columns.tolist())
print("transcript:", df_transcript.columns.tolist())
print()

# 3. 키 컬럼 존재 여부 확인
print("'customer_id' in profile:", "customer_id" in df_profile.columns)
print("'customer_id' in transcript:", "customer_id" in df_transcript.columns)
print("'offer_id' in portfolio:", "offer_id" in df_portfolio.columns)
print("'offer_id' in transcript:", "offer_id" in df_transcript.columns)
print()

# 4. 키 자료형/문자열 정리
df_profile["customer_id"] = df_profile["customer_id"].astype(str).str.strip().str.lower()
df_transcript["customer_id"] = df_transcript["customer_id"].astype(str).str.strip().str.lower()

if "offer_id" in df_portfolio.columns:
    df_portfolio["offer_id"] = df_portfolio["offer_id"].astype(str).str.strip().str.lower()

if "offer_id" in df_transcript.columns:
    df_transcript["offer_id"] = df_transcript["offer_id"].astype(str).str.strip().str.lower()

# 5. 병합 전 매칭 확인
print("transcript unique customer_id:", df_transcript["customer_id"].nunique())
print("profile unique customer_id:", df_profile["customer_id"].nunique())
print("공통 customer_id 수:", len(set(df_transcript["customer_id"]) & set(df_profile["customer_id"])))
print()

# 6. transcript + profile merge
df_merged = pd.merge(
    df_transcript,
    df_profile,
    on="customer_id",
    how="left"
)

print("=== transcript + profile merge ===")
print(df_merged.shape)
print(df_merged[["customer_id", "event"]].head())
print()

# 7. portfolio merge
df_merged = pd.merge(
    df_merged,
    df_portfolio,
    on="offer_id",
    how="left",
    suffixes=("", "_portfolio")
)

print("=== 최종 merge ===")
print(df_merged.shape)
print(df_merged.head())
print()

# 8. 불필요한 인덱스 컬럼 제거
df_merged = df_merged.drop(columns=["index", "index_portfolio"], errors="ignore")

# 9. 결측 확인
print("=== 결측치 확인 ===")
print(df_merged.isna().sum())

=== 원본 컬럼 확인 ===
profile: ['gender', 'age', 'id', 'became_member_on', 'income']
portfolio: ['reward', 'channels', 'difficulty', 'duration', 'offer_type', 'id']
transcript: ['person', 'event', 'value', 'time', 'offer_id', 'amount', 'reward_value']

=== 정리 후 컬럼 확인 ===
profile: ['gender', 'age', 'customer_id', 'became_member_on', 'income']
portfolio: ['reward', 'channels', 'difficulty', 'duration', 'offer_type', 'offer_id']
transcript: ['customer_id', 'event', 'value', 'time', 'offer_id', 'amount', 'reward_value']

'customer_id' in profile: True
'customer_id' in transcript: True
'offer_id' in portfolio: True
'offer_id' in transcript: True

transcript unique customer_id: 17000
profile unique customer_id: 14825
공통 customer_id 수: 14825

=== transcript + profile merge ===
(306137, 11)
                        customer_id           event
0  78afa995795e4d85b5d9ceeca43f5fef  offer received
1  a03223e636434f42ac4c3df47e8bac43  offer received
2  e2127556f4f64592b11af22de27a7932  offer received
3  

In [ ]:
# 이벤트별 결측 확인
df_merged.groupby("event")[["offer_id", "amount", "reward_value"]].apply(lambda x: x.isna().sum())

,offer_id,amount,reward_value
event,,,
offer completed,0,33182,0
offer received,0,76277,76277
offer viewed,0,57725,57725
transaction,138953,0,138953


In [4]:
# profile 매칭 실패 행 수 확인
print(df_merged[df_merged["became_member_on"].isna()]["customer_id"].nunique())

2175


In [5]:
# merge 성공률 확인
check_merge = pd.merge(
    df_transcript,
    df_profile,
    on="customer_id",
    how="left",
    indicator=True
)

print(check_merge["_merge"].value_counts())

_merge
both          272388
left_only      33749
right_only         0
Name: count, dtype: int64


[결론]
병합하며 생긴 결측치들은 제거하면 안 된다 => 대부분 그대로 유지하는 게 맞다

1. 지금 결측치는 “데이터 오류”가 아니라 이벤트 구조+left join 때문에 의도적으로 생긴 값

In [ ]:
# ============================================================
# 6. 통합 CSV 저장
# ============================================================
df_merged.to_csv("clean_data/starbucks_merged.csv", encoding="utf-8-sig")

print("="*60)
print("starbucks_merged.csv 저장 완료")
print("저장 경로: clean_data/starbucks_merged.csv")
print("="*60)